# HR Candidate Screening — Multi-Agent System
## Assignment 1 | LangGraph | Domain: Human Resources

**GitHub Repository:** https://github.com/Ankush-Gorade/HR-Project

This notebook clones the repository and runs the full pipeline end-to-end.
All source code lives in the GitHub repo as proper `.py` modules.

### Pipeline Overview
| Agent | Role | Tool |
|---|---|---|
| Agent 1: Input Guard | Validate & sanitise inputs | Rule-based + LLM |
| Agent 2: Resume Parser | Extract structured resume data | LLM + few-shot |
| Agent 3: JD Matcher | Score technical skill fit | **Tavily Web Search** |
| Agent 4: Behavioral Scorer | Score soft skills & culture fit | **DuckDuckGo Search** |
| Agent 5: Output Guard | Generate report, redact PII | Gmail MCP + Calendar MCP |

### Orchestration Patterns
- Conditional routing — invalid input → early reject
- Parallel fan-out/fan-in — Agents 3 & 4 run concurrently
- Human-in-the-loop — reviewer checkpoint before final output
- Iterative refinement loop — re-score on reviewer request

---
## Step 1 — Clone Repository

In [18]:
import os, subprocess, sys

REPO_URL = 'https://github.com/Ankush-Gorade/HR-Project.git'
REPO_DIR = '/content/HR-Project'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Repository cloned successfully')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print('Repository updated')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working directory: {os.getcwd()}')

# Verify all required folders exist
required = ['agents', 'pipeline', 'utils', 'prompts', 'tests', 'evaluation']
for folder in required:
    status = 'OK' if os.path.exists(folder) else 'MISSING'
    print(f'  {folder}/ : {status}')

Repository updated
Working directory: /content/HR-Project
  agents/ : OK
  pipeline/ : OK
  utils/ : OK
  prompts/ : OK
  tests/ : OK
  evaluation/ : OK


---
## Step 2 — Install Dependencies

In [19]:
!pip install -q langchain-groq langchain-community langchain langgraph
!pip install -q pydantic python-dotenv duckduckgo-search tavily-python
print('All dependencies installed')
print('NOTE: If you see import errors after this, go to Runtime -> Restart session, then run from Step 3')

All dependencies installed
NOTE: If you see import errors after this, go to Runtime -> Restart session, then run from Step 3


---
## Step 3 — Configure API Keys
We need to add Groq api key and Tavily Api key to be added in below cell

In [20]:
import os, sys

# Add repo to Python path (run this after every restart)
REPO_DIR = '/content/HR-Project'
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# Required: get free key at console.groq.com
os.environ['GROQ_API_KEY']         = ''

# Optional: enables Tavily web search in JD Matcher (app.tavily.com - free)
os.environ['TAVILY_API_KEY']       = ''

# Pipeline settings
os.environ['LANGCHAIN_TRACING_V2'] = 'false'
os.environ['HUMAN_REVIEW_ENABLED'] = 'true'

print('API keys configured')
print('Groq key set:', os.environ['GROQ_API_KEY'][:10], '...')

API keys configured
Groq key set: gsk_m5uxj2 ...


---
## Step 4 — Verify All Imports

In [21]:
from config import cfg
from utils.helpers import score_to_recommendation, weighted_average
from utils.guardrails import validate_candidate_input
from pipeline.orchestrator import run_pipeline
from agents import (
    run_input_guard,
    run_resume_parser,
    run_jd_matcher,
    run_behavioral_scorer,
    run_output_guard,
)

print('config           loaded')
print('utils            loaded')
print('pipeline         loaded')
print('All 5 agents     loaded')
print(f'Interview threshold  : {cfg.scoring.interview_threshold}')
print(f'Strong hire threshold: {cfg.scoring.strong_hire_threshold}')
print(f'Auto reject below    : {cfg.scoring.auto_reject_threshold}')

config           loaded
utils            loaded
pipeline         loaded
All 5 agents     loaded
Interview threshold  : 65
Strong hire threshold: 85
Auto reject below    : 30


---
## Step 5 — Run Full Pipeline (Happy Path)
Complete end-to-end run with a strong candidate profile.
Exercises all 5 agents, parallel scoring, human review, and both real tools.

In [22]:
import sys
mods = [k for k in sys.modules if k.startswith(('utils','agents','config','pipeline'))]
for m in mods: del sys.modules[m]

from pipeline.orchestrator import run_pipeline

print('Running HR Screening Pipeline...')
print('=' * 55)

result = run_pipeline(
    resume_text=(
        'Rahul Mehta | rahul.mehta@email.com | Pune, India\n'
        'Senior Data Engineer with 5 years experience.\n\n'
        'SKILLS\n'
        'Python, SQL, Apache Spark, Kafka, Airflow, Docker, AWS, dbt, Terraform\n\n'
        'EXPERIENCE\n'
        'Senior Data Engineer - PhonePe Bangalore (2021-Present)\n'
        '- Led team of 6 engineers, reduced latency by 40%\n'
        '- Mentored 3 junior engineers\n'
        'Data Engineer - Flipkart Bangalore (2019-2021)\n'
        '- ETL pipelines, 500GB daily data ingestion\n\n'
        'EDUCATION\n'
        'B.Tech CSE VJTI Mumbai 2019\n\n'
        'CERTIFICATIONS\n'
        'AWS Certified Data Analytics, Databricks Certified Associate'
    ),
    job_description=(
        'Senior Data Engineer needed.\n'
        'Required: Python, Apache Spark, SQL, AWS, Airflow\n'
        'Preferred: Kafka, dbt, Terraform\n'
        'Experience: 4+ years. Education: B.Tech CSE.'
    ),
    job_title='Senior Data Engineer',
    candidate_name='Rahul Mehta',
    llm_model='llama-3.1-8b-instant',
)

report = result.get('screening_report', {})
print('         HR SCREENING REPORT')
print('=' * 55)
print(f"Candidate      : {report.get('candidate_name')}")
print(f"Overall Score  : {report.get('overall_score')}")
print(f"Recommendation : {report.get('recommendation')}")
print('\nScore Breakdown:')
for k, v in report.get('score_breakdown', {}).items():
    print(f'  {k:22s}: {v}')
print('\nStrengths:')
for s in report.get('strengths', []): print(f'  - {s}')
print('\nConcerns:')
for c in report.get('concerns', []): print(f'  - {c}')
print('\nInterview Topics:')
for t in report.get('suggested_interview_topics', []): print(f'  - {t}')
print(f"\nNext Action    : {report.get('next_action')}")
print(f"Pipeline Status: {result.get('pipeline_status')}")
print('=' * 55)

Running HR Screening Pipeline...
14:46:49 [INFO] Orchestrator → Building LangGraph pipeline...


INFO:Orchestrator:Building LangGraph pipeline...


14:46:49 [INFO] Orchestrator → Pipeline compiled successfully ✔


INFO:Orchestrator:Pipeline compiled successfully ✔


14:46:49 [INFO] Orchestrator → Starting pipeline run | trace_id: 13f741f9


...

14:46:54 [INFO] OutputGuardAgent → Output guard complete ✔ — Score: 92.1, Recommendation: Strong Hire


INFO:OutputGuardAgent:Output guard complete ✔ — Score: 92.1, Recommendation: Strong Hire


14:46:54 [INFO] Orchestrator → Pipeline complete | status: completed


INFO:Orchestrator:Pipeline complete | status: completed


         HR SCREENING REPORT
Candidate      : Rahul Mehta
Overall Score  : 92.1
Recommendation : Strong Hire

Score Breakdown:
  skill_match           : 100.0
  experience_match      : 80.0
  behavioral            : 90.5
  education_match       : 100.0

Strengths:
  - Excellent match with all required skills present
  - Meets experience requirement
  - Strong leadership profile with quantified impact and mentoring experience

Concerns:

Interview Topics:
  - Walk through of the data warehouse migration project
  - Deep dive on Spark optimization techniques used at PhonePe
  - How do you mentor junior engineers?

Next Action    : Schedule technical interview immediately
Pipeline Status: completed


---
## Step 6 — Run All 5 Test Scenarios
Covers every pipeline branch:

| # | Scenario | Tests |
|---|---|---|
| 1 | Happy path — strong candidate | Full pipeline, Strong Hire |
| 2 | Weak candidate | LLM semantic validation, rejection |
| 3 | Prompt injection attack | Rule-based guardrail, early reject |
| 4 | Missing required fields | Schema validation, early reject |
| 5 | Human refinement loop | Iteration counter, re-score, approve |

In [23]:
import sys
mods = [k for k in sys.modules if k.startswith(('utils','agents','config','pipeline','tests'))]
for m in mods: del sys.modules[m]

from tests.test_scenarios import run_all_scenarios, print_summary

print('Running all 5 test scenarios...')
results = run_all_scenarios()
print_summary(results)

Running all 5 test scenarios...

🧪 Running Scenario 1: Strong Candidate (Happy Path)...
14:46:54 [INFO] Orchestrator → Building LangGraph pipeline...


INFO:Orchestrator:Building LangGraph pipeline...


14:46:54 [INFO] Orchestrator → Pipeline compiled successfully ✔


INFO:Orchestrator:Pipeline compiled successfully ✔


14:46:54 [INFO] Orchestrator → Starting pipeline run | trace_id: 0b697e9b


...

14:49:10 [INFO] OutputGuardAgent → Output guard complete ✔ — Score: 88.7, Recommendation: Strong Hire


INFO:OutputGuardAgent:Output guard complete ✔ — Score: 88.7, Recommendation: Strong Hire


14:49:10 [INFO] Orchestrator → Pipeline complete | status: completed


INFO:Orchestrator:Pipeline complete | status: completed



  SCENARIO 5 — Human Refinement Loop
  Pipeline Status : completed
  Candidate       : Priya Sharma
  Overall Score   : 88.7
  Recommendation  : Strong Hire
  Score Breakdown :
    skill_match         : 85.0
    experience_match    : 90.0
    behavioral          : 91.0
    education_match     : 95.0
  Concerns        :
    • Missing data warehousing skill from required skills
  Next Action     : Schedule technical interview immediately

  TEST SCENARIOS SUMMARY
  #    Scenario                  Status       Result          Pass
  -------------------------------------------------------
  1    Happy Path                completed    Hire            ✅ PASS
  2    Weak Candidate            rejected     Reject          ✅ PASS
  3    Prompt Injection          rejected     Reject          ✅ PASS
  4    Missing Fields            rejected     Reject          ✅ PASS
  5    Human Refinement Loop     completed    Strong Hire     ✅ PASS
  -------------------------------------------------------
  Ove

---
## Step 7 — Sub-Agent Evaluation
Standalone evaluation of **ResumeParserAgent** on 20 manually curated test cases.

**Metric:** Field-level Precision / Recall / F1  
**Dataset:** 20 diverse resumes covering fresh graduates to 10-year veterans

In [32]:
import sys, os
mods = [k for k in sys.modules if k.startswith(('utils','agents','config','pipeline','evaluation'))]
for m in mods: del sys.modules[m]

# Ensure evaluation package is importable
eval_init = '/content/HR-Project/evaluation/__init__.py'
if not os.path.exists(eval_init):
    with open(eval_init, 'w') as f:
        f.write('"""evaluation package"""')

from evaluation.eval_script import run_evaluation
results, metrics = run_evaluation()

print(f"Mean Accuracy : {metrics['overall']['mean_accuracy']:.1%}")
print(f"Mean F1 Score : {metrics['overall']['mean_f1']:.3f}")


🔬 Loading evaluation dataset from /content/HR-Project/evaluation/eval_dataset.json...
   Loaded 20 test cases

🤖 Running ResumeParserAgent on all test cases...
   (This may take 1-2 minutes)

   Processing case  1/20... 14:59:11 [INFO] ResumeParserAgent → Parsing resume...


INFO:ResumeParserAgent:Parsing resume...


14:59:12 [INFO] ResumeParserAgent → Resume parsed ✔ — Skills: 5, Experience: 4.0 years


INFO:ResumeParserAgent:Resume parsed ✔ — Skills: 5, Experience: 4.0 years


✅ 7/7 fields correct
   Processing case  2/20... 14:59:12 [INFO] ResumeParserAgent → Parsing resume...


...

✅ 7/7 fields correct
   Processing case 20/20... 15:01:46 [INFO] ResumeParserAgent → Parsing resume...


INFO:ResumeParserAgent:Parsing resume...


15:01:59 [INFO] ResumeParserAgent → Resume parsed ✔ — Skills: 6, Experience: 6.0 years


INFO:ResumeParserAgent:Resume parsed ✔ — Skills: 6, Experience: 6.0 years


✅ 7/7 fields correct

  RESUME PARSER AGENT — EVALUATION RESULTS
  ID   Name  Skills Email Exp  #Exp  Yrs  Edu  Overall
  ------------------------------------------------------------
  1      ✅     ✅      ✅    ✅     ✅    ✅    ✅   ✅
  2      ✅     ✅      ✅    ✅     ✅    ✅    ✅   ✅
  3      ✅     ✅      ✅    ✅     ✅    ✅    ✅   ✅
  4      ✅     ✅      ✅    ✅     ✅    ✅    ✅   ✅
  5      ✅     ✅      ✅    ✅     ✅    ✅    ✅   ✅
  6      ✅     ✅      ✅    ✅     ✅    ✅    ✅   ✅
  7      ✅     ✅      ✅    ✅     ✅    ✅    ✅   ✅
  8      ✅     ✅      ✅    ✅     ✅    ✅    ✅   ✅
  9      ✅     ✅      ✅    ✅     ✅    ✅    ✅   ✅
  10     ✅     ✅      ✅    ✅     ✅    ✅    ✅   ✅
  11     ✅     ✅      ✅    ✅     ✅    ✅    ✅   ✅
  12     ✅     ✅      ✅    ✅     ✅    ✅    ✅   ✅
  13     ✅     ✅      ✅    ✅     ✅    ✅    ✅   ✅
  14     ✅     ✅      ✅    ✅     ✅    ✅    ✅   ✅
  15     ✅     ✅      ✅    ✅     ✅    ✅    ✅   ✅
  16     ✅     ✅      ✅    ✅     ✅    ✅    ✅   ✅
  17     ✅     ✅      ✅    ✅     

In [33]:
!pip install -q streamlit pyngrok pandas
!pip install -q langchain-groq langchain-community langchain langgraph
!pip install -q pydantic python-dotenv duckduckgo-search
!pip install -q langgraph langchain-groq langchain-community pydantic duckduckgo-search tavily-python
print("Done")

Done


In [34]:
import os, subprocess, sys

!git clone https://github.com/Ankush-Gorade/HR-Project.git
os.chdir('/content/HR-Project')
sys.path.insert(0, '/content/HR-Project')
print("Repo ready")
!ls

fatal: destination path 'HR-Project' already exists and is not an empty directory.
Repo ready
agents	   evaluation  pipeline     README.md	      tests
config.py  HR-Project  prompts	    requirements.txt  utils
docs	   logs        __pycache__  streamlit_app.py


In [35]:
# Install necessary tunneling tool
!pip install -q streamlit pyngrok

In [36]:
import subprocess
import sys
import time
from pyngrok import ngrok

# 1. Complete Cleanup
!pkill streamlit
ngrok.kill()

# 2. Force Installation in the specific Python Path Streamlit uses
# This ensures 'langgraph' is available to the background process
print("Installing langgraph in the background environment...")
!{sys.executable} -m pip install -q langgraph langchain-groq tavily-python duckduckgo-search

# 3. Start Streamlit using the SAME executable
# We add a small delay to let the installation settle
print("Starting Streamlit...")
process = subprocess.Popen([
    sys.executable, "-m", "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.address", "localhost"
])

# 4. Wait for the server to actually start before tunneling
time.sleep(8)

# 5. Connect the tunnel
try:
    public_url = ngrok.connect(8501, proto="http")
    print(f"\n🚀 SUCCESS! Open this URL:")
    print(f"👉 {public_url}")
except Exception as e:
    print(f"❌ Tunnel Error: {e}")

Installing langgraph in the background environment...
Starting Streamlit...

🚀 SUCCESS! Open this URL:
👉 NgrokTunnel: "https://sustained-yin-appease.ngrok-free.dev" -> "http://localhost:8501"


In [37]:
import os, time
from pyngrok import ngrok

# Kill all processes
os.system('pkill -f streamlit')
os.system('pkill -f ngrok')
ngrok.kill()
time.sleep(3)

# Delete the old minimal app.py that's causing the wrong UI
for f in ['/content/HR-Project/app.py', '/content/app.py']:
    if os.path.exists(f):
        os.remove(f)
        print(f'🗑️  Deleted: {f}')

print('✅ Cleanup done')

✅ Cleanup done


##Here we need to add NGROK_TOKEN

In [30]:
import subprocess, sys, time, os
from pyngrok import ngrok, conf

NGROK_TOKEN   = ''
STREAMLIT_APP = '/content/HR-Project/streamlit_app.py'
PORT          = 8501

# Double-check right file
print(f'Launching: {STREAMLIT_APP}')
assert os.path.exists(STREAMLIT_APP)

env = os.environ.copy()

process = subprocess.Popen([
    sys.executable, '-m', 'streamlit', 'run', STREAMLIT_APP,
    '--server.port', str(PORT),
    '--server.address', 'localhost',
    '--server.headless', 'true',
    '--server.enableCORS', 'false',
    '--server.enableXsrfProtection', 'false',
], env=env)

time.sleep(12)

ngrok.set_auth_token(NGROK_TOKEN)
conf.get_default().auth_token = NGROK_TOKEN

public_url = ngrok.connect(PORT, proto='http')
print(f'\n✅ Live at: {public_url}')

Launching: /content/HR-Project/streamlit_app.py

✅ Live at: NgrokTunnel: "https://sustained-yin-appease.ngrok-free.dev" -> "http://localhost:8501"


---
## Step 8 — Download Design Document

In [31]:
import os
from google.colab import files

doc_path = '/content/HR-Project/docs/design_document.html'
if os.path.exists(doc_path):
    size = os.path.getsize(doc_path)
    print(f'Design document found ({size//1024} KB)')
    files.download(doc_path)
    print('Download started')
    print('Open in Chrome -> File -> Print -> Save as PDF')
else:
    print('Design document not found at:', doc_path)
    print('Make sure docs/design_document.html is pushed to GitHub')

Design document found (33 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started
Open in Chrome -> File -> Print -> Save as PDF


---
## Assignment Checklist

| Requirement | Implementation | Status |
|---|---|---|
| Domain-specific use case | HR candidate screening end-to-end | Done |
| 3-4 distinct processing stages | 5 sub-agents, each single-responsibility | Done |
| Input guardrails | Length, injection, schema in Agent 1 | Done |
| Output guardrails | PII redact, bias check, schema in Agent 5 | Done |
| Tool 1 — external | Tavily Web Search in JDMatcherAgent | Done |
| Tool 2 — external | DuckDuckGo Search in BehavioralScorerAgent | Done |
| Orchestration | Conditional + parallel + HITL + loop | Done |
| Framework | LangGraph StateGraph | Done |
| Modular code | Each agent in own .py file | Done |
| 5 test scenarios | All branches covered, all passed | Done |
| Sub-agent evaluation | 20-case dataset, F1 = 1.000 | Done |
| Observability | LangSmith + structured logging | Done |
| Prompt engineering | Few-shot + CoT in all 5 prompts | Done |
| Human-in-the-loop | Reviewer checkpoint implemented | Done |
| GitHub repo | All .py files in proper modules | Done |

**GitHub:** https://github.com/Ankush-Gorade/HR-Project